In [ ]:
import torch

In [ ]:
import torchvision
from PIL import Image
import random

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import numpy as np

In [ ]:
from lightweight_gan import Trainer

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Discriminator test: convergence of hinge losses

In [ ]:
def cast_list(el):
    return el if isinstance(el, list) else [el]

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


def image_to_pil(image):
    ndarr = image.mul(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).to('cpu', torch.uint8).numpy()
    im = Image.fromarray(ndarr)
    return im

def convert_image_to(img_type, image):
    if image.mode != img_type:
        return image.convert(img_type)
    return image


In [ ]:
def test(
        data = './data',
    ndata = -1,   #JEC -1 would produce an error so one should set cmd ine
    results_dir = './results',
    models_dir = './models',
    name = 'default',
    new = False,
    load_from = -1,
    image_size = 256,
    optimizer = 'adam',
    fmap_max = 512,
    transparent = False,
    greyscale = False,
    batch_size = 1,
    gradient_accumulate_every = 4,
    num_train_steps = 150000,
    learning_rate = 2e-4,
    save_every = 1000,
    evaluate_every = 1000,
    generate = False,
    generate_types = ['default', 'ema'],
    generate_interpolation = False,
    aug_test = False,
    aug_prob=None,
    aug_types=['cutout', 'translation'],
    dataset_aug_prob=0.,
    attn_res_layers = [32],
    freq_chan_attn = False,
    disc_output_size = 1,
    dual_contrast_loss = False,
    antialias = False,
    interpolation_num_steps = 100,
    save_frames = False,
    num_image_tiles = None,
    num_workers = None,
    multi_gpus = False,
    calculate_fid_every = None,
    calculate_fid_num_images = 12800,
    clear_fid_cache = False,
    seed = 42,
    amp = False,
    show_progress = False,
    use_aim = False,
    aim_repo = None,
    aim_run_hash = None,
    load_strict = True
):
    def dbg(obj,msg=""): print(msg,obj, type(obj).__name__)

    model_args = dict(
        name = name,
        results_dir = results_dir,
        models_dir = models_dir,
        batch_size = batch_size,
        gradient_accumulate_every = gradient_accumulate_every,
        attn_res_layers = cast_list(attn_res_layers),
        freq_chan_attn = freq_chan_attn,
        disc_output_size = disc_output_size,
        dual_contrast_loss = dual_contrast_loss,
        antialias = antialias,
        image_size = image_size,
        num_image_tiles = num_image_tiles,
        optimizer = optimizer,
        num_workers = num_workers,
        fmap_max = fmap_max,
        transparent = transparent,
        greyscale = greyscale,
        lr = learning_rate,
        save_every = save_every,
        evaluate_every = evaluate_every,
        aug_prob = aug_prob,
        aug_types = cast_list(aug_types),
        dataset_aug_prob = dataset_aug_prob,
        calculate_fid_every = calculate_fid_every,
        calculate_fid_num_images = calculate_fid_num_images,
        clear_fid_cache = clear_fid_cache,
        amp = amp,
        load_strict = load_strict
    )

    model = Trainer(**model_args, use_aim = use_aim)
    model.load(load_from)
    model.set_data_src(data, ndata=ndata)
    model.GAN.eval()
    return model

In [ ]:
set_seed(42)
ntrain=100_000     # size of the dataset used for training: eg. 1000, 10000, 100000
epoch=100          # epoch of optimisation  (eg. 0, 50, 100, 147 or 150)
assert int(epoch) in (0,10,50,100,147,150), "incorrect epoch number"
#for 10^5 training dataset size, the last optimization was 147
if int(ntrain) == 100_000: 
    if int(epoch)>147: 
        epoch = 147
datasetA = '../datasets/sdss_'+str(ntrain)+'_A'
datasetB = '../datasets/sdss_'+str(ntrain)+'_B'

modelA = test(data = datasetA, ndata=ntrain, aug_prob=0.0, batch_size = 1, 
              name = 'train_'+str(ntrain)+'_noaugatall_A', load_from=epoch)
modelB = test(data = datasetB, ndata=ntrain, aug_prob=0.0, batch_size = 1, 
              name = 'train_'+str(ntrain)+'_noaugatall_B', load_from=epoch)


In [ ]:
latent_dim = modelA.GAN.latent_dim
print("letent_dim:",latent_dim)

In [ ]:
import torch.nn.functional as F

def relu(x):
    return np.maximum(0, x)


def hinge_loss(real, fake):
    return (relu(1 + real) + relu(1 - fake))    


In [ ]:
all_logit_A = []
all_logit_BA = []
all_logit_A_train = []
if num > 100:
    all_fake_logit_A = []
    all_fake_logit_B = []
    latent_dim = modelA.GAN.latent_dim
    print("letent_dim:",latent_dim)
all_logit_B = []
all_logit_AB = []
all_logit_B_train = []


for i in range(1000):  # 1minute for 1000
    if i%100 == 0: 
        print(".... ",i)
    latents = torch.randn((1, latent_dim)).to(device)   # z
    generated_image_A = modelA.generate_(modelA.GAN.G, latents) # x_A = G_A(z)  
    logit_A, _,_ = modelA.GAN.D(generated_image_A)   # logit_A = D_A(x_A)
    logit_BA, _,_ = modelB.GAN.D(generated_image_A)   # logit_BA = D_B(x_A)
    train_image_A = next(modelA.loader)
    logit_A_train, _,_ = modelA.GAN.D(train_image_A.to(device))  # logit_A_train = D_A(x_train_A)

    if num > 100:
        fake_img = torch.rand_like(generated_image_A)  # x_random
        fake_logit_A, _, _ = modelA.GAN.D(fake_img)
        fake_logit_B, _, _ = modelB.GAN.D(fake_img)

    generated_image_B = modelB.generate_(modelB.GAN.G, latents) # x_B = G_B(z)
    logit_B, _,_ = modelB.GAN.D(generated_image_B)   # logit_B = D_B(x_B)
    logit_AB, _,_ = modelA.GAN.D(generated_image_B)   # logit_AB = D_A(x_B)
    train_image_B = next(modelB.loader)
    logit_B_train, _,_ = modelB.GAN.D(train_image_B.to(device))  # logit_A_train = D_A(x_train_A)

    
    all_logit_A.append(logit_A.item())
    all_logit_BA.append(logit_BA.item())
    all_logit_A_train.append(logit_A_train.item())
    if num > 100:
        all_fake_logit_A.append(fake_logit_A.item())
        all_fake_logit_B.append(fake_logit_B.item())
    all_logit_B.append(logit_B.item())
    all_logit_AB.append(logit_AB.item())
    all_logit_B_train.append(logit_B_train.item())


In [ ]:
all_logit_A = np.array(all_logit_A)
all_logit_BA = np.array(all_logit_BA)
all_logit_A_train = np.array(all_logit_A_train)
if epoch > 100:
    all_fake_logit_A = np.array(all_fake_logit_A)
    all_fake_logit_B = np.array(all_fake_logit_B)
all_logit_B = np.array(all_logit_B)
all_logit_AB = np.array(all_logit_AB)
all_logit_B_train = np.array(all_logit_B_train)

In [ ]:
# in points - start with the body text size and play around
SMALL_SIZE = 14
MEDIUM_SIZE = 16
BIGGER_SIZE = 18

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)

In [ ]:
#D_A(x_A) vs D_A(x_B)
loss_1 =  hinge_loss(all_logit_A,all_logit_AB)
loss_2 = hinge_loss(all_logit_A,all_logit_A_train)
loss_3 = hinge_loss(all_logit_B,all_logit_BA)
loss_4 = hinge_loss(all_logit_B,all_logit_B_train)
vmin = min(np.min(loss_1),np.min(loss_2),np.min(loss_3),np.min(loss_4))
vmax = max(np.max(loss_1),np.max(loss_2),np.max(loss_3),np.max(loss_4))

if epoch>100:
    loss_5 = hinge_loss(all_logit_A,all_fake_logit_A)
    vmin = min(vmin, np.min(loss_5))
    vmax = max(vmax, np.max(loss_5))


_,bins,_ = plt.hist(loss_1,bins=100,range=(vmin,vmax),
                    density=True,histtype='step',lw=3,label=r"$\ell(D_A(x_A),D_A(x_B))$");
#D_A(x_A) vs D_A(x_train)
plt.hist(loss_2,bins=bins,density=True,
        histtype='step',lw=3,label=r"$\ell(D_A(x_A),D_A(x_{train}))$",ls="-.");
plt.hist(loss_3,bins=100,density=True,
         histtype='step',lw=3,label=r"$\ell(D_B(x_B),D_B(x_A))$");
plt.hist(loss_4,bins=bins,density=True,
        histtype='step',lw=3,label=r"$\ell(D_B(x_B),D_B(x_{train}))$",ls="-.");

if epoch>150:
    plt.hist(loss_5,bins=bins,density=True,
            histtype='step',lw=3, label=r"$\ell(D_A(x_A),D_A(x_{rand}))$");

plt.legend(fontsize=13);
plt.xlabel(r"$\ell(x,y)$");
plt.title(f"epoch={epoch}")
plt.savefig("loss_discri_"+str(ntrain)+"_A_B_num_"+str(epoch)+".pdf")